# 05 Multi Head Attention

## Problem

为什么一个 attention 头通常不够？多头机制到底是在并行学什么，它解决的是表达能力问题、信息解耦问题，还是纯粹的参数扩展问题？

## Dependencies

需要已经理解单头 self-attention 的输入输出、score、weights 和 `weights @ V` 的意义。


## Goals

- 理解多头注意力如何让模型并行学习不同关系
- 理解 head splitting、concat、output projection 的完整流程
- 理解总 hidden dim 固定时，单个 head dim 是怎么分出来的
- 能解释为什么多头不是“单头复制几次”

## Scope and Approach

这一节不会把多头写成一句“拆成多个头再拼起来”就结束，而是要讲清楚为什么单头不够、多头究竟在给模型增加什么能力，以及它在 shape 上到底怎么落地。


## 为什么一个头通常不够

如果只有一个 attention 头，模型必须把很多不同类型的关系都压在同一个注意力空间里：

- 指代关系
- 句法关系
- 远距离依赖
- 位置信息
- 主题相关性

这不是说单头完全做不到，而是说很多不同关系会更容易互相干扰。多头的核心直觉是：

- 把总表示空间切成多个子空间
- 每个子空间学一套自己的 Q、K、V
- 让不同头偏向不同关注模式

所以多头更像“并行的多个视角”，而不是“一个视角变大一号”。


In [ ]:
import numpy as np

# 固定随机种子，方便反复看同一组数。
np.random.seed(1)
np.set_printoptions(precision=3, suppress=True)

# 这里构造一个小输入：4 个位置，每个位置 8 维。
x = np.random.randn(4, 8) * 0.2  # shape = (seq_len, hidden_dim)

num_heads = 2
hidden_dim = x.shape[1]

# hidden_dim 固定以后，每个 head 分到的维度就是 head_dim。
head_dim = hidden_dim // num_heads

# 先在总空间里生成 Q、K、V，再拆头。
W_q = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_k = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_v = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_o = np.random.randn(hidden_dim, hidden_dim) * 0.2

Q = x @ W_q
K = x @ W_k
V = x @ W_v

# 现在把最后一维拆成多个 head。
# Q.shape 从 (seq_len, hidden_dim) 变成 (seq_len, num_heads, head_dim)
Q = Q.reshape(len(x), num_heads, head_dim)
K = K.reshape(len(x), num_heads, head_dim)
V = V.reshape(len(x), num_heads, head_dim)

print('x.shape =', x.shape)
print('Q.shape =', Q.shape)
print('K.shape =', K.shape)
print('V.shape =', V.shape)
print('head_dim =', head_dim)


## 多头到底是怎么工作的

可以拆成四步：

1. 先对输入做线性投影，得到总的 `Q`、`K`、`V`
2. 再把最后一维拆成多个 head
3. 每个 head 各自做一遍 attention
4. 把所有 head 的输出拼回去，再做一层输出投影 `W_o`

这里一个非常关键的点是：

- 如果 `hidden_dim` 固定
- 那 head 数增加时
- 每个 head 的维度 `head_dim` 往往会变小

所以多头不是无代价地“白送更多能力”，而是在子空间拆分和表示解耦之间做权衡。


In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

all_head_outputs = []

for h in range(num_heads):
    # 取出第 h 个头自己的 q/k/v。
    q = Q[:, h, :]  # shape = (seq_len, head_dim)
    k = K[:, h, :]
    v = V[:, h, :]

    # 每个 head 都独立算一遍 attention score。
    scores = (q @ k.T) / np.sqrt(head_dim)

    # 仍然要加 causal mask，因为我们还是在 decoder-only 设定下。
    mask = np.triu(np.ones_like(scores), k=1) * -1e9
    weights = softmax(scores + mask)
    head_output = weights @ v

    all_head_outputs.append(head_output)

    print(f'head {h} weights =')
    print(weights)

# 多个 head 的输出沿最后一维拼接回来。
# 如果每个 head 输出 shape = (seq_len, head_dim)
# 拼完后 shape = (seq_len, hidden_dim)
concat = np.concatenate(all_head_outputs, axis=-1)

# 最后再经过一层输出投影，把多个头的结果重新混合。
output = concat @ W_o

print('concat.shape =', concat.shape)
print('output.shape =', output.shape)


## 为什么最后还要一个 W_o

很多人会把 `W_o` 当成可有可无的收尾层，其实它很重要。它至少做两件事：

- 把多个头拼回来的表示重新混合
- 让不同头之间的结果再次交互，而不是永远各管各的

如果没有 `W_o`，那多头的结果更像“简单并排放在一起”，而不是重新形成一个统一表示空间。


## Common mistakes

- 以为“头越多越好”。头数增加会影响 `head_dim`、参数组织和实际效率，不能无限堆。
- 忘记总 `hidden_dim` 通常是固定的，因此头数变多时每个头的维度会变小。
- 只关注最终输出，不去看每个 head 的注意力矩阵，就看不见多头真正的价值。
- 把多头理解成单头复制几份，而忽略了它是在不同子空间里学不同投影。

## Checkpoints

- 把 `num_heads` 改成 4，观察 `head_dim` 如何变化。
- 比较不同 head 的权重矩阵，看看它们是否会偏向不同位置。
- 用自己的话解释为什么多头结束后还需要 `W_o`。
- 回答：多头增加的主要是参数量，还是表达解耦能力？

## Summary

MHA 让 attention 从“一个视角看上下文”变成“多个子空间并行看上下文”。它的关键价值不是简单堆参数，而是让不同关系有机会在不同头里被分开建模，最后再重新混合回统一表示。

## Next Module

下一节进入位置编码，重点是 RoPE。因为到目前为止，attention 还只是在比较内容，它天然并不知道 token 的先后顺序。
